# Questão 8 — Sistema de Recomendação

## Objetivo
Desenvolver um motor de recomendação simples com base no comportamento de compra dos clientes, utilizando similaridade de cosseno entre produtos.

## Produto de referência
**GPS Garmin Vortex Maré Drift**

## O que será feito
1. Construir a matriz usuário × produto
2. Transformar a matriz em interações binárias:
   - 1 = cliente comprou o produto ao menos uma vez
   - 0 = cliente não comprou
3. Calcular a similaridade de cosseno entre produtos
4. Gerar o ranking dos 5 produtos mais similares ao item de referência
5. Desconsiderar o próprio GPS no ranking final



##  Conexão com o Snowflake e Leitura das tabelas

Nesta etapa, é realizada a conexão com o banco de dados Snowflake para leitura das tabelas necessárias à análise.

As tabelas utilizadas são:
- **LH_NAUTICALS.QUESTAO8.VENDAS** → contém as compras realizadas pelos clientes
- **LH_NAUTICALS.QUESTAO8.DADOS_TRATADOS** → contém os dados dos produtos

Nesta etapa, as tabelas são carregadas do Snowflake e convertidas para DataFrames do pandas, para facilitar o tratamento e a análise em Python.

In [32]:
import pandas as pd
import os

from dotenv import load_dotenv
from snowflake.snowpark import Session



load_dotenv()

conn = {
    "account": os.getenv("account"),
    "user": os.getenv("user"),
    "password": os.getenv("password"),
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "LH_NAUTICALS"
}

session = Session.builder.configs(conn).create()



df_vendas_snowpark = session.table("LH_NAUTICALS.QUESTAO8.VENDAS")
df_produtos_snowpark = session.table("LH_NAUTICALS.QUESTAO8.DADOS_TRATADOS")

df_vendas = df_vendas_snowpark.to_pandas()
df_produtos = df_produtos_snowpark.to_pandas()

print("Colunas da tabela de vendas:")
print(list(df_vendas.columns))

print("\nColunas da tabela de produtos:")
print(list(df_produtos.columns))




Colunas da tabela de vendas:
['ID', 'ID_CLIENT', 'ID_PRODUCT', 'QTD', 'TOTAL', 'SALE_DATE']

Colunas da tabela de produtos:
['PRODUCT_ID', 'PRODUCT_NAME', 'CATEGORY', 'START_DATE', 'USD_PRICE']


##  Seleção e padronização das colunas necessárias

Para resolver a questão, serão utilizadas apenas as seguintes colunas:

### Tabela de vendas
- `ID_CLIENT`
- `ID_PRODUCT`

### Tabela de produtos
- `PRODUCT_ID`
- `PRODUCT_NAME`

Também será feita a padronização dos identificadores para o tipo `string`, evitando problemas de compatibilidade entre tipos durante a criação da matriz e o cálculo da similaridade.

In [33]:
# Mantém apenas as colunas necessárias
df_vendas = df_vendas[["ID_CLIENT", "ID_PRODUCT"]].copy()
df_produtos = df_produtos[["PRODUCT_ID", "PRODUCT_NAME"]].copy()

# Remove duplicidades da tabela de produtos
df_produtos = df_produtos.drop_duplicates()

# Padroniza os tipos como string
df_vendas["ID_CLIENT"] = df_vendas["ID_CLIENT"].astype(str)
df_vendas["ID_PRODUCT"] = df_vendas["ID_PRODUCT"].astype(str)

df_produtos["PRODUCT_ID"] = df_produtos["PRODUCT_ID"].astype(str)
df_produtos["PRODUCT_NAME"] = df_produtos["PRODUCT_NAME"].astype(str).str.strip()

print("Amostra da tabela de vendas:")
print(df_vendas.head())

print("\nAmostra da tabela de produtos:")
print(df_produtos.head())

Amostra da tabela de vendas:
   ID_CLIENT  ID_PRODUCT
0  id_client  id_product
1         42         105
2          3         136
3         25         139
4         20          23

Amostra da tabela de produtos:
   PRODUCT_ID                    PRODUCT_NAME
0           1     Transponder AIS Maré Magnum
15          2       Transponder Furuno Marlin
19          3    Radar Furuno Pulse Leviathan
26          4       Rádio AIS Hydro Tidal Zen
34          5  Piloto Automático Furuno Storm


##  Identificação do produto de referência

O produto de referência definido na questão é:

**GPS Garmin Vortex Maré Drift**

Nesta etapa, será localizado o identificador (`PRODUCT_ID`) correspondente a esse item na tabela de produtos.

In [34]:
# Encontrando o produto

produto_base_nome = "GPS Garmin Vortex Maré Drift"

produto_base = df_produtos[
    df_produtos["PRODUCT_NAME"].str.contains(produto_base_nome, case=False, na=False)
]

print("Produto(s) encontrado(s):")
print(produto_base)

# Captura o ID do produto de referência
id_produto_base = produto_base.iloc[0]["PRODUCT_ID"]

print("ID do produto de referência:", id_produto_base)
print("Tipo do ID:", type(id_produto_base))

Produto(s) encontrado(s):
    PRODUCT_ID                  PRODUCT_NAME
236         27  GPS Garmin Vortex Maré Drift
ID do produto de referência: 27
Tipo do ID: <class 'str'>


## Construção da matriz usuário × produto

A questão pede uma matriz de interação em que:

- as linhas representem os clientes (`id_client`)
- as colunas representem os produtos (`id_product`)
- o valor da célula seja:
  - `1`, caso o cliente tenha comprado o produto ao menos uma vez
  - `0`, caso contrário

A quantidade comprada não será considerada, apenas a presença ou ausência da compra.

In [35]:
# Construção da matriz
# Cria uma coluna binária de interação
df_vendas["INTERACAO"] = 1

# Remove compras repetidas do mesmo cliente para o mesmo produto
interacoes = df_vendas.drop_duplicates(subset=["ID_CLIENT", "ID_PRODUCT"])

print("Amostra da base de interações binárias:")
print(interacoes.head())

# Criação da matriz usuário x produto
matriz_usuario_produto = interacoes.pivot_table(
    index="ID_CLIENT",
    columns="ID_PRODUCT",
    values="INTERACAO",
    aggfunc="max",
    fill_value=0
)

print("Dimensão da matriz usuário x produto:")
print(matriz_usuario_produto.shape)

print("\nAmostra da matriz usuário x produto:")
print(matriz_usuario_produto.head())


Amostra da base de interações binárias:
   ID_CLIENT  ID_PRODUCT  INTERACAO
0  id_client  id_product          1
1         42         105          1
2          3         136          1
3         25         139          1
4         20          23          1
Dimensão da matriz usuário x produto:
(51, 152)

Amostra da matriz usuário x produto:
ID_PRODUCT  1  10  100  101  102  103  104  105  106  107  ...  92  93  94  \
ID_CLIENT                                                  ...               
1           1   0    1    0    1    1    0    0    1    1  ...   1   1   0   
10          0   1    0    1    1    1    1    0    1    1  ...   1   0   1   
11          1   0    0    1    0    0    1    1    0    1  ...   1   1   0   
12          1   1    1    0    0    1    1    0    1    1  ...   1   1   0   
13          1   1    1    1    1    1    1    1    1    1  ...   1   1   1   

ID_PRODUCT  95  96  97  98  99  None  id_product  
ID_CLIENT                                         
1        

##  Conversão para matriz produto × usuário

Como a similaridade será calculada entre produtos, é necessário transpor a matriz anterior.

Assim:
- cada linha passa a representar um produto
- cada coluna passa a representar um cliente

Dessa forma, cada produto será representado por um vetor binário de clientes que o compraram.

In [36]:
# Conversão para matriz produto x usuário
matriz_produto_usuario = matriz_usuario_produto.T

print("Dimensão da matriz produto x usuário:")
print(matriz_produto_usuario.shape)

print("\nAmostra da matriz produto x usuário:")
print(matriz_produto_usuario.head())

Dimensão da matriz produto x usuário:
(152, 51)

Amostra da matriz produto x usuário:
ID_CLIENT   1  10  11  12  13  14  15  16  17  18  ...  47  48  49  5  6  7  \
ID_PRODUCT                                         ...                        
1           1   0   1   1   1   1   1   1   1   1  ...   1   1   1  1  1  1   
10          0   1   0   1   1   0   1   0   1   0  ...   1   1   1  1  0  1   
100         1   0   0   1   1   1   1   1   1   0  ...   1   0   1  1  1  1   
101         0   1   1   0   1   1   0   0   1   1  ...   0   1   1  1  1  0   
102         1   1   0   0   1   1   1   1   1   1  ...   1   1   0  0  0  1   

ID_CLIENT   8  9  None  id_client  
ID_PRODUCT                         
1           1  1     0          0  
10          0  1     0          0  
100         0  1     0          0  
101         1  0     0          0  
102         1  1     0          0  

[5 rows x 51 columns]


## Cálculo da similaridade de cosseno entre produtos

A similaridade de cosseno mede o quão parecidos são dois vetores.

Neste contexto:
- cada produto é representado pelo conjunto de clientes que o compraram
- produtos comprados por perfis semelhantes de clientes tendem a possuir maior similaridade

A similaridade será calculada produto × produto.

In [37]:
# Similaridade do cosseno
similaridade = cosine_similarity(matriz_produto_usuario)

similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto_usuario.index,
    columns=matriz_produto_usuario.index
)

print("Dimensão da matriz de similaridade:")
print(similaridade_df.shape)

print("\nAmostra da matriz de similaridade:")
print(similaridade_df.iloc[:5, :5])

Dimensão da matriz de similaridade:
(152, 152)

Amostra da matriz de similaridade:
ID_PRODUCT         1        10       100       101       102
ID_PRODUCT                                                  
1           1.000000  0.698771  0.726722  0.753819  0.732140
10          0.698771  1.000000  0.687500  0.697486  0.636656
100         0.726722  0.687500  1.000000  0.610300  0.636656
101         0.753819  0.697486  0.610300  1.000000  0.676661
102         0.732140  0.636656  0.636656  0.676661  1.000000


##  Verificação da presença do produto de referência na matriz

Antes de gerar o ranking, é importante validar se o produto de referência realmente está presente na matriz de similaridade.

In [38]:
# Verificando a presença do produto de referencia na matriz

print("Produto base:", id_produto_base)
print("Está presente na matriz de similaridade?", id_produto_base in similaridade_df.columns)

Produto base: 27
Está presente na matriz de similaridade? True


##  Geração do ranking de produtos mais similares

Agora será gerado o ranking de similaridade em relação ao produto de referência.

Regras aplicadas:
- considerar o produto **GPS Garmin Vortex Maré Drift** como item base
- remover o próprio GPS do ranking
- selecionar apenas os **5 produtos mais similares**

In [39]:
# Geração do ranking
# Seleciona a coluna do produto base e ordena da maior para a menor similaridade
ranking = similaridade_df[id_produto_base].sort_values(ascending=False)

# Remove o próprio produto do ranking
ranking = ranking.drop(index=id_produto_base, errors="ignore")

# Seleciona os 5 produtos mais similares
top5 = ranking.head(5).reset_index()
top5.columns = ["PRODUCT_ID", "SIMILARIDADE"]

print("Top 5 IDs mais similares ao produto base:")
print(top5)

Top 5 IDs mais similares ao produto base:
  PRODUCT_ID  SIMILARIDADE
0         94      0.869626
1         11      0.868037
2         35      0.853913
3          1      0.850000
4        115      0.850000


##  Inclusão do nome dos produtos recomendados

Para tornar o resultado interpretável, será feito um cruzamento com a tabela de produtos, trazendo o nome correspondente a cada `PRODUCT_

In [40]:
# resultado
resultado_final = top5.merge(
    df_produtos[["PRODUCT_ID", "PRODUCT_NAME"]].drop_duplicates(),
    on="PRODUCT_ID",
    how="left"
)

resultado_final = resultado_final[["PRODUCT_ID", "PRODUCT_NAME", "SIMILARIDADE"]]
resultado_final["SIMILARIDADE"] = resultado_final["SIMILARIDADE"].round(4)

print("Resultado final:")
print(resultado_final)

Resultado final:
  PRODUCT_ID                                PRODUCT_NAME  SIMILARIDADE
0         94            Motor de Popa Volvo Magnum 276HP        0.8696
1         11         GPS Furuno Swift Leviathan Poseidon        0.8680
2         35                          Radar Furuno Swift        0.8539
3          1                 Transponder AIS Maré Magnum        0.8500
4        115  Cabo de Nylon Delta Force Magnum Leviathan        0.8500


##  Conclusão

Com base na matriz usuário × produto e no cálculo da similaridade de cosseno entre os vetores dos produtos, foi possível identificar os itens mais semelhantes ao produto **GPS Garmin Vortex Maré Drift**.

Esse tipo de abordagem permite implementar uma recomendação do tipo:

**"Quem comprou isso, também levou..."**

mesmo sem o uso de ferramentas complexas de Big Data, utilizando apenas Python e bibliotecas de análise de dados.